# Parte Específica: EDA distribucional do BankChurners

Dataset tratado em `eda_unknown.ipynb`: `out/parte_especifica/bankchurners_tratado.csv`
("Unknown" mantido como categoria própria).

Pilar distribucional: como cada variável se distribui, sozinha.

In [ ]:
# [IA] Claude Code – importacao das bibliotecas, leitura do dataset tratado e pasta de saida
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Sem isso o matplotlib le o "$" das faixas de renda ("$40K - $60K") como formula matematica
plt.rcParams["text.parse_math"] = False

CAMINHO_DADOS = Path("../../out/parte_especifica/bankchurners_tratado.csv")
PASTA_SAIDA = Path("../../out/parte_especifica")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CAMINHO_DADOS)
COLUNA_ALVO = "Attrition_Flag"
print(df.shape)
df.head()

In [ ]:
# [IA] Claude Code – grupos de variaveis (CLIENTNUM e identificador, fica fora da analise)
colunas_numericas = df.select_dtypes(include="number").columns.drop("CLIENTNUM").tolist()
colunas_categoricas = df.select_dtypes(exclude="number").columns.drop(COLUNA_ALVO).tolist()

# Ordem natural das categorias ordinais (o value_counts ordenaria por frequencia)
ORDEM_CATEGORIAS = {
    "Education_Level": ["Uneducated", "High School", "College", "Graduate", "Post-Graduate", "Doctorate", "Unknown"],
    "Income_Category": ["Less than $40K", "$40K - $60K", "$60K - $80K", "$80K - $120K", "$120K +", "Unknown"],
    "Card_Category": ["Blue", "Silver", "Gold", "Platinum"],
}
print("Numericas:", colunas_numericas)
print("Categoricas:", colunas_categoricas)

## 1. Variável alvo

In [ ]:
# [IA] Claude Code – contagem e proporcao da variavel alvo
alvo = pd.DataFrame({
    "qtd": df[COLUNA_ALVO].value_counts(),
    "pct": (df[COLUNA_ALVO].value_counts(normalize=True) * 100).round(2),
})
alvo

In [ ]:
# [IA] Claude Code – grafico de barras da variavel alvo
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(alvo.index, alvo["qtd"], color=["#4C72B0", "#C44E52"])
for x, (q, p) in enumerate(zip(alvo["qtd"], alvo["pct"])):
    ax.text(x, q, f"{q} ({p:.1f}%)", ha="center", va="bottom")
ax.set_title("BankChurners: clientes ativos x clientes que cancelaram")
ax.set_xlabel("Situacao do cliente (Attrition_Flag)")
ax.set_ylabel("Quantidade de clientes")
ax.set_ylim(0, alvo["qtd"].max() * 1.15)
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "eda_distribuicao_alvo.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Variáveis numéricas

In [ ]:
# [IA] Claude Code – medidas resumo + assimetria e curtose de cada variavel numerica
resumo = df[colunas_numericas].describe().T
resumo["assimetria"] = df[colunas_numericas].skew()
resumo["curtose"] = df[colunas_numericas].kurt()
resumo["cv_pct"] = (resumo["std"] / resumo["mean"] * 100)  # coeficiente de variacao
resumo.round(3)

In [ ]:
# [IA] Claude Code – classificar a forma pela assimetria (|a| < 0.5 ~ simetrica; 0.5 a 1 moderada; > 1 forte)
def forma(a):
    if abs(a) < 0.5:
        return "aproximadamente simetrica"
    lado = "a direita" if a > 0 else "a esquerda"
    return f"assimetria {'moderada' if abs(a) <= 1 else 'forte'} {lado}"

pd.DataFrame({
    "assimetria": resumo["assimetria"].round(2),
    "forma": resumo["assimetria"].map(forma),
    "qtd_valores_distintos": df[colunas_numericas].nunique(),
}).sort_values("assimetria")

In [ ]:
# [IA] Claude Code – histogramas de todas as variaveis numericas (com media e mediana)
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
for ax, coluna in zip(axes.flat, colunas_numericas):
    discreta = df[coluna].nunique() <= 15
    sns.histplot(df[coluna], bins=30, discrete=discreta, kde=not discreta, ax=ax, color="#4C72B0")
    ax.axvline(df[coluna].mean(), color="#C44E52", linestyle="--", linewidth=1.5, label="media")
    ax.axvline(df[coluna].median(), color="#55A868", linestyle="-", linewidth=1.5, label="mediana")
    ax.set_title(coluna)
    ax.set_xlabel(coluna)
    ax.set_ylabel("Quantidade de clientes")
    ax.legend(fontsize=8)
for ax in axes.flat[len(colunas_numericas):]:
    ax.set_visible(False)
fig.suptitle("BankChurners: distribuicao das variaveis numericas", fontsize=16)
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "eda_histogramas_numericas.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# [IA] Claude Code – variaveis com assimetria forte vistas em escala log (so para leitura da forma)
fortes = resumo.index[resumo["assimetria"].abs() > 1].tolist()
fig, axes = plt.subplots(1, len(fortes), figsize=(5 * len(fortes), 4), squeeze=False)
for ax, coluna in zip(axes.flat, fortes):
    sns.histplot(np.log1p(df[coluna]), bins=30, kde=True, ax=ax, color="#8172B3")
    ax.set_title(f"log(1 + {coluna})")
    ax.set_xlabel(f"log(1 + {coluna})")
    ax.set_ylabel("Quantidade de clientes")
fig.suptitle("Variaveis com assimetria forte em escala logaritmica", fontsize=14)
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "eda_histogramas_log.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# [IA] Claude Code – concentracoes em um unico valor (picos que aparecem nos histogramas)
picos = pd.DataFrame({
    "valor_mais_frequente": df[colunas_numericas].mode().iloc[0],
    "pct_no_valor_mais_frequente": [
        (df[c] == df[c].mode().iloc[0]).mean() * 100 for c in colunas_numericas
    ],
}).round(2).sort_values("pct_no_valor_mais_frequente", ascending=False)
picos.head(8)

In [ ]:
# [IA] Claude Code – Total_Trans_Amt parece ter mais de um "morro"; ver o histograma com mais detalhe
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(df["Total_Trans_Amt"], bins=80, ax=ax, color="#4C72B0")
ax.set_title("Total_Trans_Amt: valor total transacionado em 12 meses (80 faixas)")
ax.set_xlabel("Total_Trans_Amt (US$)")
ax.set_ylabel("Quantidade de clientes")
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "eda_hist_total_trans_amt.png", dpi=150, bbox_inches="tight")
plt.show()

### Leitura das distribuições numéricas

_(texto do grupo)_

## 3. Variáveis categóricas

In [ ]:
# [IA] Claude Code – frequencias absolutas e relativas de cada variavel categorica
for coluna in colunas_categoricas:
    ordem = ORDEM_CATEGORIAS.get(coluna, df[coluna].value_counts().index.tolist())
    tabela = pd.DataFrame({
        "qtd": df[coluna].value_counts().reindex(ordem),
        "pct": (df[coluna].value_counts(normalize=True) * 100).reindex(ordem).round(2),
    })
    print(f"\n{coluna}")
    print(tabela.to_string())

In [ ]:
# [IA] Claude Code – barras horizontais das categoricas (ordinais na ordem natural; Unknown em cinza)
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, coluna in zip(axes.flat, colunas_categoricas):
    ordem = ORDEM_CATEGORIAS.get(coluna, df[coluna].value_counts().sort_values().index.tolist())
    contagem = df[coluna].value_counts().reindex(ordem)
    if coluna in ORDEM_CATEGORIAS:
        contagem = contagem.iloc[::-1]  # primeira categoria em cima
    cores = ["#B0B0B0" if c == "Unknown" else "#4C72B0" for c in contagem.index]
    ax.barh(contagem.index.astype(str), contagem.values, color=cores)
    for y, v in enumerate(contagem.values):
        ax.text(v, y, f" {v / len(df) * 100:.1f}%", va="center", fontsize=9)
    ax.set_title(coluna)
    ax.set_xlabel("Quantidade de clientes")
    ax.set_ylabel(coluna)
    ax.set_xlim(0, contagem.max() * 1.2)
for ax in axes.flat[len(colunas_categoricas):]:
    ax.set_visible(False)
fig.suptitle("BankChurners: distribuicao das variaveis categoricas (cinza = Unknown)", fontsize=15)
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "eda_barras_categoricas.png", dpi=150, bbox_inches="tight")
plt.show()

### Leitura das distribuições categóricas

_(texto do grupo)_

## Observações do pilar distribucional

_(texto do grupo)_